#ChEMBL Drug Target Extraction

Purpose:
extract drug -> target protein relationships from ChEMBL
for the selected medication list

Output:
drug_targets.csv containing:
    - ChEMBL ID
    - target name
    - UniProt accession

In [ ]:
#import packages
import requests
import pandas as pd
#print("yaaa")

yaaa


In [ ]:
#define drugs
drug_df = pd.read_csv("C:/Katieryb/Pipelines/proteins/drug_list.csv.txt")

drug_df.head()

<>:2: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
<>:2: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
C:\Users\kitty\AppData\Local\Temp\ipykernel_2808\91095152.py:2: SyntaxWarning: "\K" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\K"? A raw string is also an option.
  drug_df = pd.read_csv("C:\Katieryb\Pipelines\proteins\drug_list.csv.txt")


,Drug
0,Ibuprofen
1,Acetaminophen
2,Aspirin
3,Naproxen
4,Diphenhydramine


In [27]:
#find ChEMBL ID

def get_chembl_id(drug_name):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search?q={drug_name}&format=json"
    response = requests.get(url)
    data = response.json()
    chembl_id = data["molecules"][0]["molecule_chembl_id"]

    return chembl_id

#test:
get_chembl_id("ibuprofen")

'CHEMBL521'

In [28]:
#get targets

def get_targets(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"

    response = requests.get(url)

    data = response.json()

    targets = []

    for item in data["mechanisms"]:
        targets.append(item["target_chembl_id"])

    return targets

#test
get_targets("CHEMBL521")

['CHEMBL2094253']

In [29]:
#convert target ID to protein
def get_protein(target_id):

    url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
    response = requests.get(url)
    data = response.json()
    
    proteins = []

    for component in data["target_components"]:

        #adding this to standardize labels to connect to pathways
        gene = None

        for synonym in component["target_component_synonyms"]:
            if synonym["syn_type"] == "GENE_SYMBOL":
                gene = synonym["component_synonym"]

        proteins.append({
            "protein": component["component_description"],
            "uniprot": component["accession"],
            "gene":  gene,
            "target_type": data["target_type"]
        })

    return proteins

#test
"""
chembl_id = get_chembl_id("ibuprofen")

targets = get_targets(chembl_id)

for target in targets:
    print(get_protein(target))
"""

'\nchembl_id = get_chembl_id("ibuprofen")\n\ntargets = get_targets(chembl_id)\n\nfor target in targets:\n    print(get_protein(target))\n'

In [ ]:
#the pipeline -> queries drugs

all_results = []

for drug in drug_df["Drug"]:
    print("Processing:", drug)

    #add a test for if the drug is not valid

    chembl_id = get_chembl_id(drug)

    targets = get_targets(chembl_id)

    for target in targets:

        proteins = get_protein(target)

        for protein in proteins:

            all_results.append({

                "Drug": drug,
                "ChEMBL_ID": chembl_id,
                "Protein": protein["protein"],
                "Gene": protein["gene"],
                "UniProt": protein["uniprot"],
                "Target_Type": protein["target_type"]
            })

results = pd.DataFrame(all_results)

results

results.to_csv("C:/Katieryb/Pipelines/proteins/drug_targets.csv", index=False)
#if run from here all takes about -> 3-5 minutes (17 drugs)

#targets = pd.read_csv("C:/Katieryb/Pipelines/proteins/drug_targets.csv")



Processing: Ibuprofen
Processing: Acetaminophen
Processing: Aspirin
Processing: Naproxen
Processing: Diphenhydramine
Processing: Loratadine
Processing: Cetirizine
Processing: Amoxicillin
Processing: Azithromycin
Processing: Lisinopril
Processing: Metoprolol
Processing: Amlodipine
Processing: Metformin
Processing: Fluoxetine
Processing: Sertraline
Processing: Clozapine
Processing: Insulin


In [31]:
#debugging
results.head()

,Drug,ChEMBL_ID,Protein,Gene,UniProt,Target_Type
0,Ibuprofen,CHEMBL521,Prostaglandin G/H synthase 2,PTGS2,P35354,PROTEIN FAMILY
1,Ibuprofen,CHEMBL521,Prostaglandin G/H synthase 1,PTGS1,P23219,PROTEIN FAMILY
2,Aspirin,CHEMBL25,Prostaglandin G/H synthase 2,PTGS2,P35354,PROTEIN FAMILY
3,Aspirin,CHEMBL25,Prostaglandin G/H synthase 1,PTGS1,P23219,PROTEIN FAMILY
4,Loratadine,CHEMBL998,Histamine H1 receptor,HRH1,P35367,SINGLE PROTEIN


In [ ]:
#debugging
results.shape
results["Drug"].unique()
results.isna().sum()



<StringArray>
['P35354', 'P23219', 'P35367', 'P12821', 'P08588', 'Q13936', 'Q13698',
 'Q01668', 'O60840', 'P31645', 'P06213']
Length: 11, dtype: str

In [33]:
import os

os.path.exists("C:/Katieryb/Pipelines/proteins/drug_targets.csv")

True